In [39]:
import pandas as pd
import os

# Set the data directory path
data_dir = '../data/raw'

# Load the Fake news dataset
fake_df = pd.read_csv(os.path.join(data_dir, 'Fake.csv'))

# Load the True news dataset
true_df = pd.read_csv(os.path.join(data_dir, 'True.csv'))

# Display basic information about the datasets
print("Fake News Dataset:")
print(f"Shape: {fake_df.shape}")
print(f"\nFirst few rows:")
fake_df.head()


Fake News Dataset:
Shape: (23481, 4)

First few rows:


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [40]:
print("True News Dataset:")
print(f"Shape: {true_df.shape}")
print(f"\nFirst few rows:")
true_df.head()


True News Dataset:
Shape: (21417, 4)

First few rows:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


## DECISION : Deduplicate articles in both dataframes

In [41]:
# Deduplicate articles in both dataframes
fake_df = fake_df.drop_duplicates(subset=['title', 'text'], keep='first').reset_index(drop=True)
true_df = true_df.drop_duplicates(subset=['title', 'text'], keep='first').reset_index(drop=True)

In [42]:
print(f"Shape: {true_df.shape}")

Shape: (21197, 4)


In [43]:
print(f"Shape: {fake_df.shape}")

Shape: (17908, 4)


## DECISION : Remove publisher and the place of publication from beginning text column in both dataframes

In [44]:
# Remove publisher and place of publication from the beginning of 'text' in both dataframes
# Pattern: "PLACE (Publisher) - " or "PLACE/PLACE (Publisher) - "

import re

print("Removing publisher and place of publication from beginning of 'text' column...\n")

# Regex pattern to match: PLACE (Publisher) - 
# Examples: "WASHINGTON (Reuters) - ", "SEATTLE/WASHINGTON (Reuters) - "
pattern = r'^[A-Z][A-Z\s,/-]*\s*\([^)]+\)\s*-+\s*'

# Count before cleaning
fake_before_pattern = fake_df['text'].astype(str).str.contains(pattern, regex=True, na=False).sum()
true_before_pattern = true_df['text'].astype(str).str.contains(pattern, regex=True, na=False).sum()

print(f"Before cleaning:")
print(f"  FAKE dataset: {fake_before_pattern} texts starting with publisher/place pattern")
print(f"  TRUE dataset: {true_before_pattern} texts starting with publisher/place pattern")

# Show some examples before cleaning
print(f"\nExamples before cleaning:")
print("  FAKE dataset:")
for idx in fake_df[fake_df['text'].astype(str).str.contains(pattern, regex=True, na=False)].index[:2]:
    text = str(fake_df.loc[idx, 'text'])
    print(f"    Index {idx}: {text[:100]}...")
print("  TRUE dataset:")
for idx in true_df[true_df['text'].astype(str).str.contains(pattern, regex=True, na=False)].index[:2]:
    text = str(true_df.loc[idx, 'text'])
    print(f"    Index {idx}: {text[:100]}...")

# Apply regex replacement to remove the pattern
fake_df['text'] = fake_df['text'].astype(str).str.replace(pattern, '', regex=True).str.strip()
true_df['text'] = true_df['text'].astype(str).str.replace(pattern, '', regex=True).str.strip()

# Count after cleaning
fake_after_pattern = fake_df['text'].astype(str).str.contains(pattern, regex=True, na=False).sum()
true_after_pattern = true_df['text'].astype(str).str.contains(pattern, regex=True, na=False).sum()

print(f"\nAfter cleaning:")
print(f"  FAKE dataset: {fake_after_pattern} texts still starting with publisher/place pattern (should be 0)")
print(f"  TRUE dataset: {true_after_pattern} texts still starting with publisher/place pattern (should be 0)")

# Show some examples after cleaning
print(f"\nExamples after cleaning:")
print("  FAKE dataset:")
for idx in list(fake_df.index)[:2]:
    text = str(fake_df.loc[idx, 'text'])
    print(f"    Index {idx}: {text[:100]}...")
print("  TRUE dataset:")
for idx in list(true_df.index)[:2]:
    text = str(true_df.loc[idx, 'text'])
    print(f"    Index {idx}: {text[:100]}...")

print("\n✓ Publisher and place of publication removed from 'text' column in both datasets!")

Removing publisher and place of publication from beginning of 'text' column...

Before cleaning:
  FAKE dataset: 0 texts starting with publisher/place pattern
  TRUE dataset: 17666 texts starting with publisher/place pattern

Examples before cleaning:
  FAKE dataset:
  TRUE dataset:
    Index 0: WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted...
    Index 1: WASHINGTON (Reuters) - Transgender people will be allowed for the first time to enlist in the U.S. m...

After cleaning:
  FAKE dataset: 0 texts still starting with publisher/place pattern (should be 0)
  TRUE dataset: 124 texts still starting with publisher/place pattern (should be 0)

Examples after cleaning:
  FAKE dataset:
    Index 0: Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had...
    Index 1: House Intelligence Committee Chairman Devin Nunes is going to have a bad day. He s been under the as...
  TRUE dataset:
    Ind

In [45]:
true_df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",The head of a conservative Republican faction ...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,Transgender people will be allowed for the fir...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,The special counsel investigation of links bet...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,Trump campaign adviser George Papadopoulos tol...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,President Donald Trump called on the U.S. Post...,politicsNews,"December 29, 2017"


## DECISION : Concatenate and text columns to create new column content

In [46]:
# Concatenate title and text into a new column for both dataframes
fake_df['content'] = fake_df['title'] + ' ' + fake_df['text']
true_df['content'] = true_df['title'] + ' ' + true_df['text']

print("Fake News Dataset - New column added:")
print(fake_df[['title', 'text', 'content']].head())
print("\nTrue News Dataset - New column added:")
print(true_df[['title', 'text', 'content']].head())


Fake News Dataset - New column added:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text  \
0  Donald Trump just couldn t wish all Americans ...   
1  House Intelligence Committee Chairman Devin Nu...   
2  On Friday, it was revealed that former Milwauk...   
3  On Christmas day, Donald Trump announced that ...   
4  Pope Francis used his annual Christmas Day mes...   

                                             content  
0   Donald Trump Sends Out Embarrassing New Year’...  
1   Drunk Bragging Trump Staffer Started Russian ...  
2   Sheriff David Clarke Becomes An Internet Joke...  
3   Trump Is So Obsessed He Even Has Obama’s Name...  
4   Pope Fran

In [47]:
# Compute text length distributions
# Character length
fake_df_char_length = fake_df['content'].str.len()
true_df_char_length = true_df['content'].str.len()

# Token length (rough, using whitespace)
fake_df_token_length = fake_df['content'].str.split().str.len()
true_df_token_length = true_df['content'].str.split().str.len()

print("=== FAKE NEWS DATASET - Text Length Distribution ===\n")
print("Character Length Statistics:")
print(fake_df_char_length.describe())
print("\nToken Length Statistics:")
print(fake_df_token_length.describe())

print("\n\n=== TRUE NEWS DATASET - Text Length Distribution ===\n")
print("Character Length Statistics:")
print(true_df_char_length.describe())
print("\nToken Length Statistics:")
print(true_df_token_length.describe())


=== FAKE NEWS DATASET - Text Length Distribution ===

Character Length Statistics:
count    17908.00000
mean      2578.44818
std       2210.78003
min         30.00000
25%       1669.00000
50%       2292.00000
75%       3074.00000
max      51892.00000
Name: content, dtype: float64

Token Length Statistics:
count    17908.000000
mean       429.015077
std        357.153684
min          2.000000
25%        279.000000
50%        385.000000
75%        514.000000
max       8148.000000
Name: content, dtype: float64


=== TRUE NEWS DATASET - Text Length Distribution ===

Character Length Statistics:
count    21197.000000
mean      2424.966505
std       1684.401333
min         31.000000
25%        957.000000
50%       2261.000000
75%       3273.000000
max      29848.000000
Name: content, dtype: float64

Token Length Statistics:
count    21197.000000
mean       392.208897
std        273.931763
min          4.000000
25%        155.000000
50%        366.000000
75%        530.000000
max       5181.0

## DECISION : We cap max input length at 1000 tokens (tentative)

In [48]:
# CAPPING MAX INPUT LENGTH AT 1000 TOKENS
MAX_TOKENS = 1000

# Cap content to first 1000 tokens for both datasets
fake_df['content'] = fake_df['content'].astype(str).apply(lambda x: ' '.join(x.split()[:MAX_TOKENS]))
true_df['content'] = true_df['content'].astype(str).apply(lambda x: ' '.join(x.split()[:MAX_TOKENS]))

print("Applied max token cap ({} tokens) to FAKE and TRUE datasets.".format(MAX_TOKENS))
print("Fake news max token count:", fake_df['content'].str.split().str.len().max())
print("True news max token count:", true_df['content'].str.split().str.len().max())



Applied max token cap (1000 tokens) to FAKE and TRUE datasets.
Fake news max token count: 1000
True news max token count: 1000


In [49]:
# Check for data quality issues: Nulls, Empty strings, Very short texts

def check_data_quality(df, df_name, content_col='content'):
    """Check for nulls, empty strings, and very short texts"""
    print(f"=== {df_name.upper()} DATASET - Data Quality Check ===\n")
    
    # Check for nulls
    null_count = df[content_col].isna().sum()
    print(f"1. Null Values:")
    print(f"   Total nulls: {null_count} ({null_count/len(df)*100:.2f}%)")
    
    # Check for empty strings (after stripping whitespace)
    empty_strings = df[content_col].astype(str).str.strip().eq('').sum()
    print(f"\n2. Empty Strings (after stripping whitespace):")
    print(f"   Total empty strings: {empty_strings} ({empty_strings/len(df)*100:.2f}%)")
    
    # Check for very short texts (less than 50 characters)
    very_short_chars = (df[content_col].astype(str).str.len() < 50).sum()
    print(f"\n3. Very Short Texts:")
    print(f"   Texts with < 50 characters: {very_short_chars} ({very_short_chars/len(df)*100:.2f}%)")
    
    # Check for very short texts by token count (less than 5 tokens)
    very_short_tokens = (df[content_col].astype(str).str.split().str.len() < 5).sum()
    print(f"   Texts with < 5 tokens: {very_short_tokens} ({very_short_tokens/len(df)*100:.2f}%)")
    
    # Show examples of very short texts if any
    if very_short_chars > 0:
        print(f"\n   Examples of very short texts (< 50 chars):")
        short_examples = df[df[content_col].astype(str).str.len() < 50][content_col].head(5)
        for idx, text in enumerate(short_examples, 1):
            print(f"   {idx}. Length: {len(str(text))} chars - '{str(text)[:100]}...'")
    
    print("\n" + "="*60 + "\n")

# Check both datasets
check_data_quality(fake_df, "FAKE NEWS")
check_data_quality(true_df, "TRUE NEWS")


=== FAKE NEWS DATASET - Data Quality Check ===

1. Null Values:
   Total nulls: 0 (0.00%)

2. Empty Strings (after stripping whitespace):
   Total empty strings: 0 (0.00%)

3. Very Short Texts:
   Texts with < 50 characters: 10 (0.06%)
   Texts with < 5 tokens: 6 (0.03%)

   Examples of very short texts (< 50 chars):
   1. Length: 48 chars - 'IT’S NOT A “MUSLIM BAN” You Big Dummies! [VIDEO]...'
   2. Length: 29 chars - 'LIVE FEED: INAUGURATION 2017!...'
   3. Length: 43 chars - 'BEST TWEET OF THE WEEK: “Hillary For Sale!”...'
   4. Length: 38 chars - 'WATCH Life Accordion To Trump! [VIDEO]...'
   5. Length: 41 chars - 'YES, OBAMA…There Is A Magic Wand! [Video]...'


=== TRUE NEWS DATASET - Data Quality Check ===

1. Null Values:
   Total nulls: 0 (0.00%)

2. Empty Strings (after stripping whitespace):
   Total empty strings: 0 (0.00%)

3. Very Short Texts:
   Texts with < 50 characters: 1 (0.00%)
   Texts with < 5 tokens: 1 (0.00%)

   Examples of very short texts (< 50 chars):
   1. L

## DECISION: We remove empty or extremely short samples

In [50]:

fake_df = fake_df[fake_df['content'].str.len() > 50]
true_df = true_df[true_df['content'].str.len() > 50]

# Manual inspection: 5 random REAL samples and 5 random FAKE samples

In [51]:
# Manual inspection: 5 random REAL samples and 5 random FAKE samples
# Check for: Label issues, Clickbait-style language, Source mentions

import random

# Set seed for reproducibility (optional - remove for different samples each time)
random.seed(42)

# Sample 5 random TRUE news articles
print("="*80)
print("MANUAL INSPECTION: 5 RANDOM REAL (TRUE) NEWS SAMPLES")
print("="*80)
print("\nCheck for:")
print("- Label issues (mislabeling)")
print("- Clickbait-style language")
print("- Source mentions (Reuters, Associated Press, etc.)")
print("\n" + "="*80 + "\n")

true_samples = true_df.sample(n=5, random_state=42)

for idx, (row_idx, row) in enumerate(true_samples.iterrows(), 1):
    print(f"\n{'='*80}")
    print(f"SAMPLE {idx} (Index: {row_idx})")
    print(f"{'='*80}")
    print(f"\nSubject: {row['subject']}")
    print(f"Date: {row['date']}")
    print(f"\n--- TITLE ---")
    print(row['title'])
    print(f"\n--- TEXT (First 500 chars) ---")
    print(row['text'][:500] + "..." if len(row['text']) > 500 else row['text'])
    print(f"\n--- FULL TEXT LENGTH: {len(row['text'])} characters ---")

# Sample 5 random FAKE news articles
print("\n\n" + "="*80)
print("MANUAL INSPECTION: 5 RANDOM FAKE NEWS SAMPLES")
print("="*80)
print("\nCheck for:")
print("- Label issues (mislabeling)")
print("- Clickbait-style language")
print("- Source mentions")
print("\n" + "="*80 + "\n")

fake_samples = fake_df.sample(n=5, random_state=42)

for idx, (row_idx, row) in enumerate(fake_samples.iterrows(), 1):
    print(f"\n{'='*80}")
    print(f"SAMPLE {idx} (Index: {row_idx})")
    print(f"{'='*80}")
    print(f"\nSubject: {row['subject']}")
    print(f"Date: {row['date']}")
    print(f"\n--- TITLE ---")
    print(row['title'])
    print(f"\n--- TEXT (First 500 chars) ---")
    print(row['text'][:500] + "..." if len(row['text']) > 500 else row['text'])
    print(f"\n--- FULL TEXT LENGTH: {len(row['text'])} characters ---")


MANUAL INSPECTION: 5 RANDOM REAL (TRUE) NEWS SAMPLES

Check for:
- Label issues (mislabeling)
- Clickbait-style language
- Source mentions (Reuters, Associated Press, etc.)



SAMPLE 1 (Index: 1056)

Subject: politicsNews
Date: October 24, 2017 

--- TITLE ---
Illinois governor enlists counterparts to attack Democrats in election ad

--- TEXT (First 500 chars) ---
Illinois’ Republican governor has launched an unusual attack ad in his re-election campaign that features three neighboring governors crediting their states’ job gains to Illinois taxes crafted by a state House Democratic leader. Illinois Republican Governor Bruce Rauner, who kicked off his 2018 re-election campaign this week, saw his first term marred by a political impasse that left Illinois without a budget for two years before lawmakers overrode his veto this summer to raise taxes.     Gover...

--- FULL TEXT LENGTH: 2827 characters ---

SAMPLE 2 (Index: 6224)

Subject: politicsNews
Date: January 18, 2017 

--- TITLE ---


In [52]:
# Subject distribution in both datasets

print("="*80)
print("SUBJECT DISTRIBUTION ANALYSIS")
print("="*80)

# Fake news dataset subject distribution
print("\n" + "="*80)
print("FAKE NEWS DATASET - Subject Distribution")
print("="*80)
fake_subject_dist = fake_df['subject'].value_counts().sort_index()
fake_subject_pct = fake_df['subject'].value_counts(normalize=True).sort_index() * 100

fake_subject_df = pd.DataFrame({
    'Count': fake_subject_dist,
    'Percentage': fake_subject_pct
})
print(fake_subject_df)
print(f"\nTotal subjects: {len(fake_subject_dist)}")
print(f"Total articles: {fake_subject_dist.sum()}")

# True news dataset subject distribution
print("\n" + "="*80)
print("TRUE NEWS DATASET - Subject Distribution")
print("="*80)
true_subject_dist = true_df['subject'].value_counts().sort_index()
true_subject_pct = true_df['subject'].value_counts(normalize=True).sort_index() * 100

true_subject_df = pd.DataFrame({
    'Count': true_subject_dist,
    'Percentage': true_subject_pct
})
print(true_subject_df)
print(f"\nTotal subjects: {len(true_subject_dist)}")
print(f"Total articles: {true_subject_dist.sum()}")

# Compare subjects between datasets
print("\n" + "="*80)
print("SUBJECT COMPARISON")
print("="*80)
fake_subjects = set(fake_df['subject'].unique())
true_subjects = set(true_df['subject'].unique())

print(f"\nUnique subjects in FAKE dataset: {len(fake_subjects)}")
print(f"Unique subjects in TRUE dataset: {len(true_subjects)}")
print(f"\nSubjects only in FAKE dataset: {fake_subjects - true_subjects}")
print(f"Subjects only in TRUE dataset: {true_subjects - fake_subjects}")
print(f"Common subjects: {fake_subjects & true_subjects}")


SUBJECT DISTRIBUTION ANALYSIS

FAKE NEWS DATASET - Subject Distribution
                 Count  Percentage
subject                           
Government News    531    2.966978
News              9050   50.567134
US_News            783    4.375035
left-news          705    3.939208
politics          6828   38.151646

Total subjects: 5
Total articles: 17897

TRUE NEWS DATASET - Subject Distribution
              Count  Percentage
subject                        
politicsNews  11216   52.915644
worldnews      9980   47.084356

Total subjects: 2
Total articles: 21196

SUBJECT COMPARISON

Unique subjects in FAKE dataset: 5
Unique subjects in TRUE dataset: 2

Subjects only in FAKE dataset: {'politics', 'left-news', 'News', 'US_News', 'Government News'}
Subjects only in TRUE dataset: {'politicsNews', 'worldnews'}
Common subjects: set()


## DECISION : Replace 'politicsNews' with 'politics' in both dataframes

In [53]:
# Replace 'politicsNews' with 'politics' in both dataframes
print("Replacing 'politicsNews' with 'politics' in both datasets...\n")

# Count before replacement
fake_before = (fake_df['subject'] == 'politicsNews').sum()
true_before = (true_df['subject'] == 'politicsNews').sum()

print(f"Before replacement:")
print(f"  FAKE dataset: {fake_before} rows with 'politicsNews'")
print(f"  TRUE dataset: {true_before} rows with 'politicsNews'")

# Perform replacement
fake_df['subject'] = fake_df['subject'].replace('politicsNews', 'politics')
true_df['subject'] = true_df['subject'].replace('politicsNews', 'politics')

# Count after replacement
fake_after = (fake_df['subject'] == 'politicsNews').sum()
true_after = (true_df['subject'] == 'politicsNews').sum()
fake_politics = (fake_df['subject'] == 'politics').sum()
true_politics = (true_df['subject'] == 'politics').sum()

print(f"\nAfter replacement:")
print(f"  FAKE dataset: {fake_after} rows with 'politicsNews' (should be 0)")
print(f"  TRUE dataset: {true_after} rows with 'politicsNews' (should be 0)")
print(f"  FAKE dataset: {fake_politics} rows with 'politics'")
print(f"  TRUE dataset: {true_politics} rows with 'politics'")
print("\n✓ Replacement completed!")


Replacing 'politicsNews' with 'politics' in both datasets...

Before replacement:
  FAKE dataset: 0 rows with 'politicsNews'
  TRUE dataset: 11216 rows with 'politicsNews'

After replacement:
  FAKE dataset: 0 rows with 'politicsNews' (should be 0)
  TRUE dataset: 0 rows with 'politicsNews' (should be 0)
  FAKE dataset: 6828 rows with 'politics'
  TRUE dataset: 11216 rows with 'politics'

✓ Replacement completed!


In [54]:
# Data Leakage Check
# Confirm: No explicit label words like "fake", "hoax" inside text
# No trivial cues

import re

print("="*80)
print("DATA LEAKAGE CHECK")
print("="*80)
print("\nChecking for explicit label words and trivial cues in text content...\n")

# Define label-related words to check for (case-insensitive)
label_words = ['fake', 'hoax', 'false news', 'fake news', 'not real', 
               'untrue', 'misinformation', 'disinformation', 'true news',
               'real news', 'factual', 'verified', 'legitimate']

def check_leakage(df, df_name, content_col='content'):
    """Check for label leakage words in the content"""
    print(f"{'='*80}")
    print(f"{df_name.upper()} DATASET - Leakage Check")
    print(f"{'='*80}\n")
    
    # Convert to lowercase for case-insensitive matching
    content_lower = df[content_col].astype(str).str.lower()
    
    # Check for each label word
    leakage_found = {}
    for word in label_words:
        # Use word boundaries to avoid partial matches
        pattern = r'\b' + re.escape(word.lower()) + r'\b'
        matches = content_lower.str.contains(pattern, regex=True, na=False)
        count = matches.sum()
        if count > 0:
            leakage_found[word] = {
                'count': count,
                'percentage': (count / len(df)) * 100,
                'indices': df[matches].index.tolist()[:5]  # Store first 5 indices for examples
            }
    
    # Report findings
    if leakage_found:
        print("⚠️  WARNING: Label leakage words found!\n")
        for word, info in leakage_found.items():
            print(f"Word: '{word}'")
            print(f"  Occurrences: {info['count']} ({info['percentage']:.2f}% of dataset)")
            print(f"  First few examples (indices): {info['indices'][:5]}")
            
            # Show sample text snippets
            print(f"  Sample snippets:")
            for idx in info['indices'][:3]:
                text = str(df.loc[idx, content_col])
                # Find the word in context (50 chars before and after)
                pattern = r'\b' + re.escape(word.lower()) + r'\b'
                match = re.search(pattern, text.lower())
                if match:
                    start = max(0, match.start() - 50)
                    end = min(len(text), match.end() + 50)
                    snippet = text[start:end]
                    print(f"    Index {idx}: ...{snippet}...")
            print()
    else:
        print("✓ No explicit label leakage words found in the dataset.\n")
    
    # Check for trivial cues (very short texts, repetitive patterns, etc.)
    print("Checking for other trivial cues:")
    
    # Very repetitive content (same word repeated many times)
    # This is a simple check - you might want to add more sophisticated checks
    print(f"  - Very short texts (< 50 chars): {(df[content_col].astype(str).str.len() < 50).sum()}")
    
    return leakage_found

# Check both datasets
fake_leakage = check_leakage(fake_df, "FAKE NEWS")
print()
true_leakage = check_leakage(true_df, "TRUE NEWS")

# Summary
print("="*80)
print("LEAKAGE CHECK SUMMARY")
print("="*80)
if fake_leakage or true_leakage:
    print("\n⚠️  Leakage detected! Consider cleaning the data before training.")
    print("   Label-related words found in text may cause the model to rely on trivial cues.")
else:
    print("\n✓ No significant leakage detected. Dataset looks clean for training.")


DATA LEAKAGE CHECK

Checking for explicit label words and trivial cues in text content...

FAKE NEWS DATASET - Leakage Check

⚠️  WARNING: Label leakage words found!

Word: 'fake'
  Occurrences: 1046 (5.84% of dataset)
  First few examples (indices): [0, 2, 19, 33, 39]
  Sample snippets:
    Index 0: ...out to his enemies, haters and the very dishonest fake news media. The former reality show star had just...
    Index 2: ...the FBI to see the exchanges.Clarke is calling it fake news even though copies of the search warrant are...
    Index 19: ..., 2017Austin, TX. #IStandWithMueller. Cronyn is a fake representative. He represents his own interests a...

Word: 'hoax'
  Occurrences: 134 (0.75% of dataset)
  First few examples (indices): [208, 326, 409, 521, 628]
  Sample snippets:
    Index 208: ...election, a thing which Donald Trump has called a hoax. Ultimately this assault won t succeed, but forei...
    Index 326: ...is frustrations for the millionth time:The Russia hoax continues,

## Decision: How to Handle Leakage Check Results

Based on the leakage check results, here's how to interpret and handle the findings:

### Key Observations:

1. **Context Matters**: Most instances appear to be legitimate content:
   - **Quotes/Discussions**: Many "fake news" mentions are quotes or discussions ABOUT fake news (e.g., "Trump called it fake news", "the fake news media")
   - **Journalistic Terms**: Words like "verified", "legitimate", "factual" are standard journalistic vocabulary
   - **Topic Discussion**: Real news articles often discuss fake news as a topic

2. **Distribution Analysis**:
   - FAKE dataset: 6.20% contain "fake", 3.47% contain "fake news"
   - TRUE dataset: 1.35% contain "fake", 0.73% contain "fake news"
   - This asymmetry could be problematic if the model learns to rely on these words

### Recommended Approach:

**Option 1: Keep as-is (RECOMMENDED for most cases)**
- **Pros**: Preserves natural language patterns, model learns contextual understanding
- **Cons**: Risk of model learning shortcuts
- **When to use**: If you want the model to learn nuanced language patterns and context

**Option 2: Remove label words (Conservative approach)**
- **Pros**: Reduces risk of trivial cues, forces model to learn deeper patterns
- **Cons**: Loses natural language, may hurt model performance on real-world data
- **When to use**: If you're concerned about model relying on shortcuts

**Option 3: Keep but monitor (Balanced approach)**
- Keep the data as-is but:
  - Monitor model attention weights to see if it over-relies on these words
  - Use techniques like adversarial validation
  - Test on out-of-domain data where these patterns may differ

### Decision Framework:

1. **Check if words appear in similar contexts in both datasets** → If yes, likely NOT leakage
2. **Check frequency asymmetry** → Large differences (like "fake": 6.20% vs 1.35%) suggest potential leakage
3. **Manual inspection of examples** → Verify if words are used contextually or as labels
4. **Consider your use case** → Real-world articles will contain these words naturally

### Recommendation:

**Keep the data as-is** because:
- The words appear in legitimate contexts (quotes, discussions, journalistic language)
- Removing them would artificially sanitize the text
- A good model should learn context, not just keywords
- Monitor model behavior during training to ensure it's not overfitting to these patterns


In [55]:
# Optional: Context analysis to help make decision
# Check how "fake news" appears in both datasets - in quotes vs. as direct label

print("="*80)
print("CONTEXT ANALYSIS: How 'fake news' is used in both datasets")
print("="*80)

def analyze_context(df, df_name, phrase="fake news", n_samples=10):
    """Analyze how a phrase is used in context"""
    print(f"\n{df_name.upper()} Dataset:")
    print("-" * 80)
    
    # Find occurrences (case-insensitive)
    pattern = r'\b' + re.escape(phrase) + r'\b'
    matches = df['content'].astype(str).str.contains(pattern, case=False, regex=True, na=False)
    matched_indices = df[matches].index[:n_samples]
    
    print(f"Total occurrences of '{phrase}': {matches.sum()}")
    print(f"\nSample contexts (first {len(matched_indices)} examples):\n")
    
    for idx in matched_indices:
        text = str(df.loc[idx, 'content'])
        # Find the phrase and show context
        match_obj = re.search(pattern, text, re.IGNORECASE)
        if match_obj:
            start = max(0, match_obj.start() - 100)
            end = min(len(text), match_obj.end() + 100)
            context = text[start:end]
            # Highlight the phrase
            highlighted = re.sub(pattern, f"[{phrase.upper()}]", context, flags=re.IGNORECASE)
            print(f"Index {idx}: ...{highlighted}...\n")

# Analyze "fake news" context in both datasets
analyze_context(fake_df, "FAKE NEWS", "fake news", n_samples=5)
print("\n")
analyze_context(true_df, "TRUE NEWS", "fake news", n_samples=5)

print("\n" + "="*80)
print("INTERPRETATION:")
print("="*80)
print("""
Look for patterns:
- Quotes (e.g., "Trump called it fake news") = Contextual, OK to keep
- Direct claims (e.g., "This is fake news") = Potential leakage
- Discussions (e.g., "the fake news problem") = Contextual, OK to keep
- Label-like usage = Leakage, consider removing

If most occurrences are contextual quotes/discussions, keeping the data is reasonable.
If many are direct label claims, consider removing those words.
""")


CONTEXT ANALYSIS: How 'fake news' is used in both datasets

FAKE NEWS Dataset:
--------------------------------------------------------------------------------
Total occurrences of 'fake news': 584

Sample contexts (first 5 examples):

Index 0: ...leave it at that. Instead, he had to give a shout out to his enemies, haters and the very dishonest [FAKE NEWS] media. The former reality show star had just one job to do and he couldn t do it. As our Country ra...

Index 2: ...d, and now, a search warrant has been executed by the FBI to see the exchanges.Clarke is calling it [FAKE NEWS] even though copies of the search warrant are on the Internet. I am UNINTIMIDATED by lib media attem...

Index 39: ...es Comey to call him a liar. I never asked Comey to stop investigating Flynn, he tweeted. Just more [FAKE NEWS] covering another Comey lie! I never asked Comey to stop investigating Flynn. Just more [FAKE NEWS] co...

Index 40: ...ector, Trump called him a liar, writing, I never asked Comey to 

## FINAL DECISION: Keep Data As-Is ✓

### Analysis of Context Examples:

**All examples are contextual uses, NOT direct label claims:**

1. **FAKE NEWS Dataset examples:**
   - "the very dishonest [FAKE NEWS] media" → Quote from Trump
   - "Clarke is calling it [FAKE NEWS]" → Reporting what someone said
   - "Just more [FAKE NEWS] covering..." → Quote from Trump's tweet
   - "Donald Trump retweeted [FAKE NEWS] videos" → Reporting an action
   
2. **TRUE NEWS Dataset examples:**
   - "@realDonaldTrump : - While the [FAKE NEWS] loves..." → Direct quote from Trump's tweet
   - "which the [FAKE NEWS] Media is desperate..." → Quote from Trump
   - "U.S. President Donald Trump regularly derides...as '[FAKE NEWS]'" → Reporting Trump's behavior
   - "[FAKE NEWS]!" → Direct quote from Trump

### Key Findings:

✅ **100% of sampled examples are contextual** - quotes, reporting, or discussions about fake news as a topic  
✅ **No direct label claims found** - no instances of "This article is fake news"  
✅ **Usage pattern is consistent** - both datasets use "fake news" similarly (in quotes/reporting)  
✅ **Natural language pattern** - real news articles naturally quote politicians and discuss topics  

### Final Decision: **KEEP DATA AS-IS**

**Rationale:**
1. All occurrences are legitimate contextual uses (quotes, reporting, discussions)
2. Removing these would sanitize the data unnaturally
3. The model should learn to understand context, not just keywords
4. Real-world articles will contain these words in similar contexts
5. The frequency asymmetry (584 vs 155) likely reflects that fake news articles more often quote Trump's "fake news" comments, which is itself a characteristic feature worth learning

### Recommended Next Steps:

1. **Proceed with training** using the data as-is
2. **Monitor during training:**
   - Check if model over-relies on these words
   - Use attention visualization to see what the model focuses on
   - Monitor validation performance
3. **Test on out-of-domain data** to ensure generalization
4. **Consider feature importance analysis** post-training to verify the model learns beyond keywords


In [56]:
true_df.head()

,title,text,subject,date,content
0,"As U.S. budget fight looms, Republicans flip t...",The head of a conservative Republican faction ...,politics,"December 31, 2017","As U.S. budget fight looms, Republicans flip t..."
1,U.S. military to accept transgender recruits o...,Transgender people will be allowed for the fir...,politics,"December 29, 2017",U.S. military to accept transgender recruits o...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,The special counsel investigation of links bet...,politics,"December 31, 2017",Senior U.S. Republican senator: 'Let Mr. Muell...
3,FBI Russia probe helped by Australian diplomat...,Trump campaign adviser George Papadopoulos tol...,politics,"December 30, 2017",FBI Russia probe helped by Australian diplomat...
4,Trump wants Postal Service to charge 'much mor...,President Donald Trump called on the U.S. Post...,politics,"December 29, 2017",Trump wants Postal Service to charge 'much mor...


In [57]:
true_df.describe()

,title,text,subject,date,content
count,21196,21196,21196,21196,21196
unique,20825,21190,2,716,21192
top,Factbox: Trump fills top jobs for his administ...,"The battle for the city of Raqqa, which Islami...",politics,"December 6, 2017",Timeline: Zika's origin and global spread The ...
freq,14,2,11216,166,3


In [ ]:

fake_df.describe()

,title,text,subject,date,content
count,17897,17897,17897,17897,17897
unique,17892,17450,5,1681,17894
top,MEDIA IGNORES Time That Bill Clinton FIRED His...,,News,"May 26, 2016",GEORGE SOROS BOARD MEMBER Is Chairman Of Firm ...
freq,3,435,9050,35,2


In [59]:
fake_df.head()

,title,text,subject,date,content
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",Donald Trump Sends Out Embarrassing New Year’s...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",Drunk Bragging Trump Staffer Started Russian C...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",Sheriff David Clarke Becomes An Internet Joke ...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",Trump Is So Obsessed He Even Has Obama’s Name ...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",Pope Francis Just Called Out Donald Trump Duri...


In [60]:
# Export both dataframes to processed data directory
import os

# Create processed directory if it doesn't exist
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

# Export fake news dataframe
fake_df.to_csv(os.path.join(processed_dir, 'Fake_processed.csv'), index=False)
print(f"✓ Exported fake_df to {processed_dir}/Fake_processed.csv")
print(f"  Shape: {fake_df.shape}")

# Export true news dataframe
true_df.to_csv(os.path.join(processed_dir, 'True_processed.csv'), index=False)
print(f"✓ Exported true_df to {processed_dir}/True_processed.csv")
print(f"  Shape: {true_df.shape}")

print("\n✅ Both dataframes exported successfully!")

✓ Exported fake_df to ../data/processed/Fake_processed.csv
  Shape: (17897, 5)
✓ Exported true_df to ../data/processed/True_processed.csv
  Shape: (21196, 5)

✅ Both dataframes exported successfully!
